In [2]:
import torch

In [3]:
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)

In [4]:
def T(g):
    return g * g.abs()

In [5]:
def register_hooks(optimizer: torch.optim.Optimizer):
    saved = {}
    def pre(optimizer, args, kwargs):
        with torch.no_grad():
            for group in optimizer.param_groups:
                for p in group["params"]:
                    if p.grad is not None:
                        saved[p] = p.grad.clone()
                        p.grad.copy_(T(p.grad)) # g -> g|g|

    def post(optimizer, args, kwargs):
        with torch.no_grad():
            for p, g in saved.items():
                p.grad.copy_(g)
        saved.clear()
    
    optimizer.register_step_pre_hook(pre)
    optimizer.register_step_post_hook(post)

In [ ]:
def make_model():
    torch.manual_seed(1234)
    return torch.nn.Sequential(
        torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1)
    ).double()

torch.manual_seed(42)
X = torch.randn(16, 4, dtype=torch.float64)
Y = torch.randn(16, 1, dtype=torch.float64)
 
 
def loss_fn(model):
    return ((model(X) - Y) ** 2).mean()
 
 
def make_opt(name, params):
    if name == "sgd":
        return torch.optim.SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-2)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=1e-2, weight_decay=1e-2)
    raise ValueError(name)


In [ ]:
def run(opt_name, n_steps=5):
    # with hooks
    m_hook = make_model()
    o_hook = make_opt(opt_name, m_hook.parameters())
    register_hooks(o_hook)
 
    # manual reference
    m_ref = make_model()
    o_ref = make_opt(opt_name, m_ref.parameters())
 
    # control (no transform)
    m_ctl = make_model()
    o_ctl = make_opt(opt_name, m_ctl.parameters())
 
    for step in range(n_steps):
        # hooked
        o_hook.zero_grad()
        loss_fn(m_hook).backward()
        orig_grads = [p.grad.clone() for p in m_hook.parameters()]
        o_hook.step()
        # check grads restored
        for p, g in zip(m_hook.parameters(), orig_grads):
            assert torch.allclose(p.grad, g, rtol=1e-12, atol=1e-12), \
                f"[{opt_name}] grad not restored at step {step}"
 
        # reference
        o_ref.zero_grad()
        loss_fn(m_ref).backward()
        with torch.no_grad():
            for p in m_ref.parameters():
                p.grad.copy_(T(p.grad))
        o_ref.step()
 
        # control
        o_ctl.zero_grad()
        loss_fn(m_ctl).backward()
        o_ctl.step()
 
    # check params match reference
    for p_h, p_r in zip(m_hook.parameters(), m_ref.parameters()):
        assert torch.allclose(p_h, p_r, rtol=1e-12, atol=1e-12), f"[{opt_name}] params differ from reference"
 
    # see if transform had an effect
    diff = max((p_h - p_c).abs().max().item()
               for p_h, p_c in zip(m_hook.parameters(), m_ctl.parameters()))
    assert diff > 1e-6, f"[{opt_name}] transform had no effect"

    print(f"ok  {opt_name:6s} "
          f"final loss={loss_fn(m_hook).item():.6f}  "
          f"max|hooked-control|={diff:.3e}")
 

In [12]:
for opt_name in ("sgd", "adamw"):
    # for restore in ("stash", "inverse"):
        run(opt_name)


ok  sgd    final loss=1.005059  max|hooked-control|=2.227e-01
ok  adamw  final loss=1.135877  max|hooked-control|=6.462e-03
